In [ ]:
import argparse
import json
import math
import time
import sys
from typing import Optional
 
try:
    import requests
except ImportError:
    print("[ERROR] Thiếu thư viện 'requests'. Chạy: pip install requests")
    sys.exit(1)
 
try:
    from tabulate import tabulate
    HAS_TABULATE = True
except ImportError:
    HAS_TABULATE = False
 
try:
    from colorama import Fore, Style, init as colorama_init
    colorama_init(autoreset=True)
    HAS_COLOR = True
except ImportError:
    HAS_COLOR = False

In [1]:
GROUND_TRUTH = [
    {
        "query_id": "Q01",
        "query": "What is deep learning and how do neural networks work?",
        "relevant_urls": [
            "https://en.wikipedia.org/wiki/Deep_learning",
            "https://en.wikipedia.org/wiki/Neural_network_(biology)",
            "https://en.wikipedia.org/wiki/Long_short-term_memory",
            "https://en.wikipedia.org/wiki/Residual_neural_network",
        ],
    },
    {
        "query_id": "Q02",
        "query": "Geoffrey Hinton contributions to artificial intelligence",
        "relevant_urls": [
            "https://en.wikipedia.org/wiki/Geoffrey_Hinton",
            "https://en.wikipedia.org/wiki/Deep_learning",
            "https://en.wikipedia.org/wiki/Workplace_impact_of_artificial_intelligence",
            "https://en.wikipedia.org/wiki/Open_letter_on_artificial_intelligence",
        ],
    },
    {
        "query_id": "Q03",
        "query": "How does gradient descent optimization work in machine learning?",
        "relevant_urls": [
            "https://en.wikipedia.org/wiki/Gradient_descent",
            "https://en.wikipedia.org/wiki/Comparison_of_optimization_software",
            "https://en.wikipedia.org/wiki/Deep_learning",
        ],
    },
    {
        "query_id": "Q04",
        "query": "Long short-term memory LSTM sequence modeling",
        "relevant_urls": [
            "https://en.wikipedia.org/wiki/Long_short-term_memory",
            "https://en.wikipedia.org/wiki/Deep_learning",
            "https://en.wikipedia.org/wiki/Residual_neural_network",
        ],
    },
    {
        "query_id": "Q05",
        "query": "Residual neural network skip connections image recognition",
        "relevant_urls": [
            "https://en.wikipedia.org/wiki/Residual_neural_network",
            "https://en.wikipedia.org/wiki/Deep_learning",
        ],
    },
    {
        "query_id": "Q06",
        "query": "Impact of AI on employment and workplace automation",
        "relevant_urls": [
            "https://en.wikipedia.org/wiki/Workplace_impact_of_artificial_intelligence",
            "https://en.wikipedia.org/wiki/Open_letter_on_artificial_intelligence",
            "https://en.wikipedia.org/wiki/Unemployment",
            "https://en.wikipedia.org/wiki/Post-work_society",
        ],
    },
    {
        "query_id": "Q07",
        "query": "Probability distribution statistics random variables",
        "relevant_urls": [
            "https://en.wikipedia.org/wiki/Probability_distribution",
        ],
    },
    {
        "query_id": "Q08",
        "query": "Zero-sum game theory Nash equilibrium",
        "relevant_urls": [
            "https://en.wikipedia.org/wiki/Zero-sum_game",
        ],
    },
    {
        "query_id": "Q09",
        "query": "Open source software movement free software",
        "relevant_urls": [
            "https://en.wikipedia.org/wiki/Open-source_software_movement",
            "https://en.wikipedia.org/wiki/Open_knowledge",
        ],
    },
    {
        "query_id": "Q10",
        "query": "Unemployment benefits guaranteed minimum income job guarantee",
        "relevant_urls": [
            "https://en.wikipedia.org/wiki/Unemployment",
            "https://en.wikipedia.org/wiki/Unemployment_benefits",
            "https://en.wikipedia.org/wiki/Guaranteed_minimum_income",
            "https://en.wikipedia.org/wiki/Job_guarantee",
            "https://en.wikipedia.org/wiki/Workfare",
            "https://en.wikipedia.org/wiki/Youth_unemployment",
        ],
    },
    {
        "query_id": "Q11",
        "query": "Nootropic cognitive enhancement brain supplements",
        "relevant_urls": [
            "https://en.wikipedia.org/wiki/Nootropic",
        ],
    },
    {
        "query_id": "Q12",
        "query": "Whistleblowing ethics corporate misconduct reporting",
        "relevant_urls": [
            "https://en.wikipedia.org/wiki/Whistleblowing",
        ],
    },
    {
        "query_id": "Q13",
        "query": "Working poor wage inequality poverty employment",
        "relevant_urls": [
            "https://en.wikipedia.org/wiki/Working_poor",
            "https://en.wikipedia.org/wiki/Wage_compression",
            "https://en.wikipedia.org/wiki/Credentialism_and_degree_inflation",
        ],
    },
    {
        "query_id": "Q14",
        "query": "Slow movement culture work-life balance presenteeism",
        "relevant_urls": [
            "https://en.wikipedia.org/wiki/Slow_movement_(culture)",
            "https://en.wikipedia.org/wiki/Presenteeism",
            "https://en.wikipedia.org/wiki/Post-work_society",
            "https://en.wikipedia.org/wiki/Sunday_scaries",
        ],
    },
    {
        "query_id": "Q15",
        "query": "bioRxiv ChemRxiv preprint scientific publishing",
        "relevant_urls": [
            "https://en.wikipedia.org/wiki/BioRxiv",
            "https://en.wikipedia.org/wiki/ChemRxiv",
        ],
    },
    {
        "query_id": "Q16",
        "query": "Terry Sejnowski computational neuroscience neural networks",
        "relevant_urls": [
            "https://en.wikipedia.org/wiki/Terry_Sejnowski",
            "https://en.wikipedia.org/wiki/Peter_Dayan",
            "https://en.wikipedia.org/wiki/Geoffrey_Hinton",
        ],
    },
    {
        "query_id": "Q17",
        "query": "Bullshit jobs meaningless work post-work society",
        "relevant_urls": [
            "https://en.wikipedia.org/wiki/Bullshit_Jobs",
            "https://en.wikipedia.org/wiki/Post-work_society",
            "https://en.wikipedia.org/wiki/Make-work_job",
            "https://en.wikipedia.org/wiki/Busy_work",
        ],
    },
    {
        "query_id": "Q18",
        "query": "Going postal workplace violence stress burnout",
        "relevant_urls": [
            "https://en.wikipedia.org/wiki/Going_postal",
            "https://en.wikipedia.org/wiki/Stress_leave",
            "https://en.wikipedia.org/wiki/Evil_corporation",
        ],
    },
    {
        "query_id": "Q19",
        "query": "Right to work labor unions employment law",
        "relevant_urls": [
            "https://en.wikipedia.org/wiki/Right_to_work",
            "https://en.wikipedia.org/wiki/Comprehensive_Employment_and_Training_Act",
            "https://en.wikipedia.org/wiki/Works_Progress_Administration",
        ],
    },
    {
        "query_id": "Q20",
        "query": "Peter Dayan reward learning neuroscience",
        "relevant_urls": [
            "https://en.wikipedia.org/wiki/Peter_Dayan",
            "https://en.wikipedia.org/wiki/Terry_Sejnowski",
            "https://en.wikipedia.org/wiki/Geoffrey_Hinton",
            "https://en.wikipedia.org/wiki/Neuroscientist",
        ],
    },
]

In [ ]:
def cprint(text: str, color: Optional[str] = None):
    """In màu nếu colorama khả dụng."""
    if HAS_COLOR and color:
        print(color + text + Style.RESET_ALL)
    else:
        print(text)
 
 
def normalize_url(url: str) -> str:
    """Chuẩn hoá URL để so sánh (bỏ trailing slash, lower case)."""
    return url.strip().rstrip("/").lower()
 
 
def search_api(query: str, k: int, base_url: str) -> list[dict]:
    """
    Gọi API backend và trả về danh sách kết quả.
    Mỗi phần tử: {"rank": int, "title": str, "url": str, ...}
    """
    endpoint = f"{base_url}/search"
    params = {"q": query, "k": k}
    response = requests.get(endpoint, params=params, timeout=30)
    response.raise_for_status()
    data = response.json()
    return data.get("data", [])
 
 
def precision_at_k(retrieved_urls: list[str], relevant_urls: list[str], k: int) -> float:
    """
    Precision@K = số kết quả đúng trong K kết quả đầu / K
    """
    relevant_set = {normalize_url(u) for u in relevant_urls}
    top_k = retrieved_urls[:k]
    hits = sum(1 for url in top_k if normalize_url(url) in relevant_set)
    return hits / k if k > 0 else 0.0
 
 
def recall_at_k(retrieved_urls: list[str], relevant_urls: list[str], k: int) -> float:
    """
    Recall@K = số kết quả đúng trong K kết quả đầu / tổng số kết quả đúng
    """
    relevant_set = {normalize_url(u) for u in relevant_urls}
    top_k = retrieved_urls[:k]
    hits = sum(1 for url in top_k if normalize_url(url) in relevant_set)
    return hits / len(relevant_set) if relevant_set else 0.0
 
 
def average_precision(retrieved_urls: list[str], relevant_urls: list[str]) -> float:
    """
    Average Precision (AP) cho một truy vấn.
    AP = Σ [Precision@i × rel(i)] / |relevant|
    rel(i) = 1 nếu kết quả thứ i là đúng, ngược lại = 0
    """
    relevant_set = {normalize_url(u) for u in relevant_urls}
    if not relevant_set:
        return 0.0
    hits = 0
    precision_sum = 0.0
    for i, url in enumerate(retrieved_urls, start=1):
        if normalize_url(url) in relevant_set:
            hits += 1
            precision_sum += hits / i
    return precision_sum / len(relevant_set)
 
 
def reciprocal_rank(retrieved_urls: list[str], relevant_urls: list[str]) -> float:
    """
    Reciprocal Rank (RR) = 1 / vị_trí_kết_quả_đúng_đầu_tiên
    Nếu không tìm thấy → 0
    """
    relevant_set = {normalize_url(u) for u in relevant_urls}
    for rank, url in enumerate(retrieved_urls, start=1):
        if normalize_url(url) in relevant_set:
            return 1.0 / rank
    return 0.0

In [ ]:
def run_evaluation(k: int = 10, base_url: str = "http://127.0.0.1:8000",
                   output_file: Optional[str] = None, verbose: bool = True):
 
    cprint("\n" + "═" * 65, Fore.CYAN if HAS_COLOR else None)
    cprint("   MODULE 5 – ĐÁNH GIÁ HỆ THỐNG TÌM KIẾM WIKISEARCH", Fore.CYAN if HAS_COLOR else None)
    cprint("═" * 65 + "\n", Fore.CYAN if HAS_COLOR else None)
 
    # ── Kiểm tra kết nối backend ─────────────────────────────────────────────
    try:
        health = requests.get(f"{base_url}/health", timeout=5).json()
        cprint(f"[✓] Backend kết nối thành công – {health.get('collection_count', '?')} chunks, "
               f"{health.get('full_text_index_size', '?')} bài viết", Fore.GREEN if HAS_COLOR else None)
    except Exception as e:
        cprint(f"[✗] Không kết nối được backend tại {base_url}", Fore.RED if HAS_COLOR else None)
        cprint(f"    Lỗi: {e}", Fore.RED if HAS_COLOR else None)
        cprint("\n    → Hãy chạy: uvicorn src.backend.main:app --reload", None)
        sys.exit(1)
 
    print(f"\n  Số truy vấn: {len(GROUND_TRUTH)}")
    print(f"  Đánh giá tại K = {k}")
    print(f"  Backend URL : {base_url}\n")
 
    results = []
    per_query_rows = []
 
    total_p_at_k  = 0.0
    total_ap      = 0.0
    total_rr      = 0.0
    total_recall  = 0.0
    n_queries     = len(GROUND_TRUTH)
 
    for gt in GROUND_TRUTH:
        qid   = gt["query_id"]
        query = gt["query"]
        relevant_urls = gt["relevant_urls"]
 
        if verbose:
            print(f"  [{qid}] {query[:60]}{'…' if len(query) > 60 else ''}")
 
        try:
            t0 = time.time()
            api_results = search_api(query, k=k, base_url=base_url)
            elapsed_ms  = round((time.time() - t0) * 1000, 1)
 
            retrieved_urls = [r["url"] for r in api_results]
 
            p_at_k  = precision_at_k(retrieved_urls, relevant_urls, k)
            ap      = average_precision(retrieved_urls, relevant_urls)
            rr      = reciprocal_rank(retrieved_urls, relevant_urls)
            rec_at_k = recall_at_k(retrieved_urls, relevant_urls, k)
 
            total_p_at_k  += p_at_k
            total_ap      += ap
            total_rr      += rr
            total_recall  += rec_at_k
 
            per_query_rows.append({
                "Query ID":       qid,
                "Query":          query[:45] + "…" if len(query) > 45 else query,
                f"P@{k}":         f"{p_at_k:.4f}",
                "AP":             f"{ap:.4f}",
                "RR":             f"{rr:.4f}",
                f"Recall@{k}":    f"{rec_at_k:.4f}",
                "Time(ms)":       elapsed_ms,
            })
 
            results.append({
                "query_id": qid,
                "query": query,
                "retrieved_urls": retrieved_urls,
                "relevant_urls": relevant_urls,
                "metrics": {
                    f"precision_at_{k}": round(p_at_k, 6),
                    "average_precision":  round(ap, 6),
                    "reciprocal_rank":    round(rr, 6),
                    f"recall_at_{k}":     round(rec_at_k, 6),
                    "response_time_ms":   elapsed_ms,
                }
            })
 
        except Exception as e:
            cprint(f"    [!] Lỗi khi truy vấn '{qid}': {e}", Fore.RED if HAS_COLOR else None)
            results.append({"query_id": qid, "query": query, "error": str(e)})
            n_queries -= 1
 
    # ── Tổng hợp kết quả ─────────────────────────────────────────────────────
    map_score    = total_ap     / n_queries if n_queries else 0.0
    mrr_score    = total_rr     / n_queries if n_queries else 0.0
    mean_p_at_k  = total_p_at_k / n_queries if n_queries else 0.0
    mean_recall  = total_recall  / n_queries if n_queries else 0.0
 
    # ── In bảng chi tiết ─────────────────────────────────────────────────────
    print()
    cprint("─" * 65, Fore.YELLOW if HAS_COLOR else None)
    cprint("  KẾT QUẢ TỪNG TRUY VẤN", Fore.YELLOW if HAS_COLOR else None)
    cprint("─" * 65, Fore.YELLOW if HAS_COLOR else None)
 
    if HAS_TABULATE and per_query_rows:
        headers = list(per_query_rows[0].keys())
        rows    = [list(r.values()) for r in per_query_rows]
        print(tabulate(rows, headers=headers, tablefmt="rounded_outline"))
    else:
        for r in per_query_rows:
            print("  " + " | ".join(f"{k}: {v}" for k, v in r.items()))
 
    # ── In tóm tắt tổng hợp ──────────────────────────────────────────────────
    print()
    cprint("═" * 65, Fore.CYAN if HAS_COLOR else None)
    cprint("  TỔNG HỢP ĐÁNH GIÁ HỆ THỐNG", Fore.CYAN if HAS_COLOR else None)
    cprint("═" * 65, Fore.CYAN if HAS_COLOR else None)
 
    summary_rows = [
        ["Số truy vấn đánh giá",         f"{n_queries} / {len(GROUND_TRUTH)}"],
        [f"Mean Precision@{k} (MP@{k})",  f"{mean_p_at_k:.4f}"],
        ["MAP  (Mean Average Precision)", f"{map_score:.4f}"],
        ["MRR  (Mean Reciprocal Rank)",   f"{mrr_score:.4f}"],
        [f"Mean Recall@{k}",              f"{mean_recall:.4f}"],
    ]
 
    if HAS_TABULATE:
        print(tabulate(summary_rows, headers=["Chỉ số", "Giá trị"],
                       tablefmt="rounded_outline"))
    else:
        for row in summary_rows:
            print(f"  {row[0]:<38}: {row[1]}")
 
    # ── Đánh giá định tính ────────────────────────────────────────────────────
    print()
    cprint("  NHẬN XÉT:", Fore.CYAN if HAS_COLOR else None)
 
    thresholds = {
        "MAP":       [(0.5, "Tốt ✓"), (0.3, "Trung bình ◎"), (0, "Cần cải thiện ✗")],
        f"P@{k}":   [(0.4, "Tốt ✓"), (0.2, "Trung bình ◎"), (0, "Cần cải thiện ✗")],
    }
 
    def rate(val, thresh_list):
        for threshold, label in thresh_list:
            if val >= threshold:
                return label
        return "N/A"
 
    print(f"  MAP  = {map_score:.4f}  → {rate(map_score, thresholds['MAP'])}")
    print(f"  MP@{k} = {mean_p_at_k:.4f}  → {rate(mean_p_at_k, thresholds[f'P@{k}'])}")
    print(f"  MRR  = {mrr_score:.4f}  → {'Tốt ✓' if mrr_score >= 0.5 else 'Trung bình ◎' if mrr_score >= 0.3 else 'Cần cải thiện ✗'}")
 
    # ── Lưu kết quả ra file JSON ──────────────────────────────────────────────
    output_data = {
        "evaluation_config": {
            "k": k,
            "backend_url": base_url,
            "n_queries": n_queries,
        },
        "aggregate_metrics": {
            f"mean_precision_at_{k}": round(mean_p_at_k, 6),
            "MAP":  round(map_score, 6),
            "MRR":  round(mrr_score, 6),
            f"mean_recall_at_{k}": round(mean_recall, 6),
        },
        "per_query_results": results,
    }
 
    if output_file:
        with open(output_file, "w", encoding="utf-8") as f:
            json.dump(output_data, f, ensure_ascii=False, indent=2)
        cprint(f"\n  [✓] Đã lưu kết quả chi tiết ra: {output_file}", Fore.GREEN if HAS_COLOR else None)
 
    cprint("\n" + "═" * 65 + "\n", Fore.CYAN if HAS_COLOR else None)
    return output_data

In [ ]:
if __name__ == "__main__":
    parser = argparse.ArgumentParser(
        description="Module 5 – Đánh giá hệ thống WikiSearch"
    )
    parser.add_argument(
        "--k", type=int, default=10,
        help="Giá trị K cho Precision@K và Recall@K (mặc định: 10)"
    )
    parser.add_argument(
        "--url", type=str, default="http://127.0.0.1:8000",
        help="URL backend API (mặc định: http://127.0.0.1:8000)"
    )
    parser.add_argument(
        "--output", type=str, default=None,
        help="Đường dẫn file JSON để lưu kết quả (tuỳ chọn)"
    )
    parser.add_argument(
        "--quiet", action="store_true",
        help="Không in chi tiết từng truy vấn"
    )
 
    args = parser.parse_args()
    run_evaluation(
        k=args.k,
        base_url=args.url,
        output_file=args.output,
        verbose=not args.quiet,
    )